# 03 · Dual-stream routing dynamics

The CSV records Fast/Slow usage telemetry. Quantify per-seed step-1→step-300 shifts and visualize trajectories. This is routing telemetry, not a reproduction of the manuscript's complete dual-stream architecture.

In [ ]:
from pathlib import Path
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project():
    candidates=[Path.cwd(), Path.cwd()/"project_inputs", Path.cwd().parent/"project_inputs", Path("/mnt/data/diina_reproducibility/project_inputs"), Path("/mnt/data")]
    for p in candidates:
        if (p/"raw_results.csv").exists(): return p
    raise FileNotFoundError("Place notebook with project_inputs/ or set cwd to extracted project directory")
PROJECT=find_project()
df=pd.read_csv(PROJECT/"raw_results.csv")
summary=json.loads((PROJECT/"summary.json").read_text())
print("Project files:",PROJECT,"| telemetry shape:",df.shape,"| columns:",list(df.columns))
assert len(df)==1500 and set(df.columns)=={"seed","step","loss","fast_usage","slow_usage","entropy"}

In [ ]:
ordered=df.sort_values("step")
ends=ordered.groupby("seed").agg(fast_start=("fast_usage","first"),fast_end=("fast_usage","last"),slow_start=("slow_usage","first"),slow_end=("slow_usage","last"))
ends["fast_shift"]=ends.fast_end-ends.fast_start; ends["slow_shift"]=ends.slow_end-ends.slow_start
display(ends); print("Mean initial/final fast:",ends.fast_start.mean(),ends.fast_end.mean()); print("Mean shifts:",ends.fast_shift.mean(),ends.slow_shift.mean())
assert np.allclose(ends.fast_shift,-ends.slow_shift,atol=1e-5)

In [ ]:
agg=ordered.groupby("step").agg(**{c:(c,"mean") for c in ["fast_usage","slow_usage"]},fast_sd=("fast_usage","std"),slow_sd=("slow_usage","std")); x=agg.index.to_numpy(); n=5; t=2.7764451051977987
fig,ax=plt.subplots(figsize=(9,4))
for name,color in [("fast_usage","#3875a7"),("slow_usage","#d17a22")]:
    y=agg[name].to_numpy(); band=agg[name.replace("usage","sd")].to_numpy()*t/np.sqrt(n)
    ax.plot(x,y,label=name.replace("_"," "),color=color); ax.fill_between(x,y-band,y+band,color=color,alpha=.18)
ax.set(xlabel="step",ylabel="utilization",ylim=(0,1),title="Fast vs Slow path allocation (mean ± 95% CI)"); ax.legend(); plt.show()

A shift is the final observed value minus the initial observed value; no interpolation or hidden routing is inferred.